# Exp 37: Plateau thresholds

Card: [exp37_plateau_thresholds.md](../../docs/neuroslope/experiments/exp37_plateau_thresholds.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_curve

from training.config import CLASSES, ROOT
from training.config import VIP
from training.cv import CVResult
from analysis.thresholds import best_f1, cross_fold_f1, plateau_f1
from analysis.plots import save_figures, use_style

CONFIG_DIR = ROOT / "configs" / "models"
MODELS = {
    "exp26 ResNet-18": CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml",
    "EfficientNet-B0": CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.yaml",
    "RegNetY-016": CONFIG_DIR / "exp33_regnety_016" / "regnety_016.yaml",
    "ConvNeXt-Tiny": CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.yaml",
}
MARGIN = 0.05  # plateau: F1 >= max - MARGIN
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp37_plateau_thresholds"

# dataviz reference palette, light mode
INK, PEAK, MIDDLE = "#52514e", "#eb6834", "#2a78d6"
use_style()

results = {name: CVResult.load(c) for name, c in MODELS.items()}

## F1 against threshold


In [ ]:
fig, axes = plt.subplots(
    len(MODELS),
    len(VIP),
    figsize=(3.4 * len(VIP), 2.8 * len(MODELS)),
    sharex=True,
    sharey=True,
    squeeze=False,
)
for (name, r), row in zip(results.items(), axes):
    peak, plateau = best_f1(r.labels, r.probs, CLASSES), plateau_f1(r.labels, r.probs, CLASSES, MARGIN)
    for cls, ax in zip(VIP, row):  # Cat, Dog: flat F1 over wide threshold range
        c = CLASSES.index(cls)
        precision, recall, thr = precision_recall_curve(r.labels[:, c], r.probs[:, c])
        f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
        ax.axvspan(
            plateau.loc[cls, "low"],
            plateau.loc[cls, "high"],
            color=MIDDLE,
            alpha=0.12,
            lw=0,
            label=f"F1 within {MARGIN} of max",
        )
        ax.plot(thr, f1, color=INK, lw=1.5)
        best_t, mean_t = peak.loc[cls, "threshold"], plateau.loc[cls, "threshold"]
        ax.axvline(best_t, color=PEAK, lw=2, ls="--", label="best_t (peak)")
        ax.axvline(mean_t, color=MIDDLE, lw=2, label="mean_t (plateau middle)")
        # value labels beside lines at two heights; left of line when near right edge
        for t, label, y in [(best_t, "best_t", 0.22), (mean_t, "mean_t", 0.08)]:
            right = t < 0.6
            ax.text(
                t + (0.015 if right else -0.015),
                y,
                f"{label} {t:.3f}",
                ha="left" if right else "right",
                fontsize=8,
                color=INK,
            )
        ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
        ax.set(xlim=(0, 1), ylim=(0, 1.05), xticks=ticks, yticks=ticks)
        ax.tick_params(labelbottom=True, labelleft=True, labelsize=8)
        if ax in axes[0]:
            ax.set_title(cls)
        if ax in axes[-1]:
            ax.set_xlabel("threshold")
    row[0].set_ylabel(f"{name}\nF1")
fig.suptitle("F1 against threshold, out-of-fold rows of folds 0–4", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.legend(
    *axes[0, 0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.96),
)
figures = {f"{PREFIX}_f1_curves.png": fig}
plt.show()

## Peak vs plateau thresholds


In [ ]:
RULES = {"peak": None, "plateau": MARGIN}  # cross_fold_f1 margin: None = best F1

table = pd.concat(
    {
        name: pd.concat(
            {
                "peak": best_f1(r.labels, r.probs, CLASSES)[["threshold", "f1"]],
                "plateau": plateau_f1(r.labels, r.probs, CLASSES, MARGIN)[["threshold", "f1"]],
            },
            axis=1,
        )
        for name, r in results.items()
    }
)
honest = pd.DataFrame(
    {
        rule: {
            name: cross_fold_f1(r.labels, r.probs, r.oof["fold"], CLASSES, margin).mean()
            for name, r in results.items()
        }
        for rule, margin in RULES.items()
    }
)
display(table.round(3), honest.rename_axis("honest Macro F1").round(3))

In [ ]:
save_figures(figures, FIGURES)